<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/module-03-assignment-03-inference-parameter-tuning-lab-template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 Assignment 3: Inference Parameter Tuning Lab

**Notebook:** Student Template  
**Runtime:** Jupyter with Ollama  
**Required model:** `gemma4:e2b`

This notebook uses short generation tasks and staged fictional résumé materials to examine how sampling settings and prompt structure affect output variability. You will compare repeated outputs, describe observable differences, and connect those differences to appropriate business uses.

> **Responsible-use boundary:** Any résumé score or hire/no-hire language produced by the model is evidence about model behavior—not a valid employment assessment or decision.


## How to Use This Notebook

> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis to write.

Work from top to bottom and leave all generated outputs visible. Complete every assigned `TODO` in the code and reflection cells. Use only the fictional materials supplied with this notebook; never substitute a real applicant résumé or other sensitive employment data.


In [ ]:
import subprocess
import time

# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

In [ ]:
from datetime import datetime
from hashlib import sha256
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

In [ ]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

### Helper Functions

These functions send requests to Ollama and display the results with run metadata. **Run this cell without changing it.**

In [ ]:
# RUN THIS CELL
def require_finished(label, value):
    """Stop before a run when required student work is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def require_options(label, options):
    """Validate that a configuration contains usable values."""
    required = {"temperature", "top_p", "top_k", "num_ctx", "num_predict"}
    missing = required.difference(options)
    if missing:
        raise ValueError(f"{label} is missing: {sorted(missing)}")
    unfinished = [key for key, value in options.items() if value is None]
    if unfinished:
        raise ValueError(f"Complete {label}: {unfinished}")


def ollama_request(path, payload=None, timeout=120):
    """Send a JSON request to the local Ollama service."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def compose_prompt(instruction, source):
    return instruction.strip() + "\n\nSOURCE\n------\n" + source.strip()


def run_once(model, prompt, run_key, options):
    """Run one independent request and preserve settings plus observable metadata."""
    require_options(f"options for {run_key}", options)
    started_at = datetime.now().astimezone().isoformat(timespec="seconds")
    start = perf_counter()
    #print(f"options:{options}")
    response = ollama_request(
        "/api/generate",
        {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "keep_alive": 300,
            "options": options,
        },
        timeout=900,
    )
    return {
        "model": model,
        "run_key": run_key,
        "recorded_at": started_at,
        "elapsed_seconds": round(perf_counter() - start, 2),
        "options": dict(options),
        "content": response["response"].strip(),
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "response": response,
    }


def display_record(record):
    option_text = ", ".join(
        f"{key}={value}" for key, value in record["options"].items()
    )
    display(Markdown(
        f"### {record['run_key']}\n\n"
        f"**Model:** `{record['model']}`  \n"
        f"**Settings:** `{option_text}`  \n"
        f"**Recorded:** {record['recorded_at']}  \n"
        f"**Elapsed:** {record['elapsed_seconds']} seconds  \n"
        f"**Prompt tokens:** {record['prompt_eval_count'] or 'n/a'}  \n"
        f"**Output tokens:** {record['eval_count'] or 'n/a'}"
    ))
    display(Markdown(record["content"]))


def run_sweep(model, prompt, parameter, values, fixed_options, prefix):
    """Run a one-factor-at-a-time sweep."""
    records = {}
    for value in values:
        options = {**fixed_options, parameter: value}
        key = f"{prefix}_{value}"
        print(f"Running {key}")
        records[key] = run_once(model, prompt, key, options)
        display_record(records[key])
    return records


### Check the Local Runtime

This check confirms that the `ollama` command is installed and available from the notebook environment.


In [ ]:
# RUN THIS CELL
import shutil

if shutil.which("ollama") is None:
    raise RuntimeError("Ollama is not installed or is not on PATH.")
print("Ollama command is available.")


In [ ]:
# RUN THIS CELL


from datetime import datetime
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json

from IPython.display import Markdown, display

AUTO_PULL_MODELS = True
OLLAMA_BASE_URL = "http://localhost:11434"

REQUIRED_MODELS = ["gemma4:e2b"]
PRIMARY_MODEL = "gemma4:e2b"
TRANSFER_MODEL = None  # This notebook uses one required model and has no transfer test.


print(f"Required models: {', '.join(REQUIRED_MODELS)}")


In [ ]:
import os

# Create the directory for module3 files if it doesn't exist
output_dir = 'module3'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f'Created directory: {output_dir}')
else:
    print(f'Directory {output_dir} already exists. Skipping creation.')

In [ ]:
# Download cto_job_posting.md
!wget -q -O module3/cto_job_posting.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/cto_job_posting.md

# Download resume_1_elena_martinez.md
!wget -q -O module3/resume_1_elena_martinez.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/resume_1_elena_martinez.md

# Download resume_2_marcus_reed.md
!wget -q -O module3/resume_2_marcus_reed.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/resume_2_marcus_reed.md

# Download resume_3_olivia_grant.md
!wget -q -O module3/resume_3_olivia_grant.md https://raw.githubusercontent.com/matthewpecsok/IS4490_student_course_files/main/module3/resume_3_olivia_grant.md

print("Downloaded all required markdown files to the 'module3' directory.")

In [ ]:
import os

# Clone the repository if it doesn't already exist
repo_dir = 'IS4490_student_course_files'
if not os.path.exists(repo_dir):
    !git clone https://github.com/matthewpecsok/IS4490_student_course_files.git
else:
    print(f'Repository {repo_dir} already exists. Skipping clone.')


In [ ]:
from pathlib import Path

data_folder = Path('IS4490_student_course_files/module3')
CTO_JOB_POSTING = (data_folder / "cto_job_posting.md").read_text(encoding="utf-8")
RESUME_ELENA = (data_folder / "resume_1_elena_martinez.md").read_text(encoding="utf-8")
RESUME_MARCUS = (data_folder / "resume_2_marcus_reed.md").read_text(encoding="utf-8")
RESUME_OLIVIA = (data_folder / "resume_3_olivia_grant.md").read_text(encoding="utf-8")
print("Loaded the staged fictional job posting and three résumés.")

In [ ]:
# RUN THIS CELL
from pathlib import Path

data_folder = Path("module3")
CTO_JOB_POSTING = (data_folder / "cto_job_posting.md").read_text(encoding="utf-8")
RESUME_ELENA = (data_folder / "resume_1_elena_martinez.md").read_text(encoding="utf-8")
RESUME_MARCUS = (data_folder / "resume_2_marcus_reed.md").read_text(encoding="utf-8")
RESUME_OLIVIA = (data_folder / "resume_3_olivia_grant.md").read_text(encoding="utf-8")
print("Loaded the staged fictional job posting and three résumés.")

In [ ]:
# RUN THIS CELL
version_info = ollama_request("/api/version")
installed = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

for model in REQUIRED_MODELS:
    if model in installed:
        print(f"Ready: {model}")
        continue
    if not AUTO_PULL_MODELS:
        raise RuntimeError(
            f"{model} is missing. Run `ollama pull {model}` or set "
            "AUTO_PULL_MODELS = True."
        )
    print(f"Downloading {model}; this one-time step may take several minutes.")
    ollama_request(
        "/api/pull",
        {"model": model, "stream": False},
        timeout=3600,
    )
    print(f"Ready: {model}")


### Confirm the Required Model

Run the next cell to verify that the required model is available before beginning the experiments.


In [ ]:
# RUN THIS CELL
installed = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
for model in REQUIRED_MODELS:
    if model not in installed:
        raise RuntimeError(f"Required model is not ready: {model}")
    print(f"Ready: {model}")


You should see:

`Ready: gemma4:e2b`


In [ ]:
# RUN THIS CELL
gpu_check = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
) if shutil.which("nvidia-smi") else None

if gpu_check and gpu_check.returncode == 0:
    print(gpu_check.stdout)
else:
    print("No NVIDIA GPU report is available; Ollama may be using CPU or another accelerator.")


# Part 1: Compare Output Variability

A language model generates text by selecting one token at a time from a set of possible next tokens. Sampling settings influence how narrow or broad that set is and how strongly the model favors likely choices.

The short prompts in this section make differences across repeated runs easier to inspect. These are demonstrations, not fully controlled one-factor-at-a-time experiments: when a comparison changes more than one setting, describe the effect of the **combined configuration** rather than claiming that one parameter caused the difference.



## Example 1: Two-Sentence Story

Run the same two-sentence story prompt three times with each configuration. The first block uses a relatively broad sampling configuration. The second combines temperature `0.0` with a very restrictive `top_p` value, which should usually produce less variation.

The first request may take longer because Ollama may need to load the model into memory. Depending on your computer, inference may use a CPU, GPU, or another accelerator. To inspect the current Ollama process, run `ollama ps` in a terminal; systems with an NVIDIA GPU can also report device use with `nvidia-smi`.

Do not assume that another student's runtime or memory observation will match yours. Hardware, model loading, software versions, and background activity can all affect the result.

This first prompt will take a while to run, likely 1-2 minutes.

Review the options to learn how to control model output. Focus on top p,k, and temperature.

In [ ]:
# RUN THIS CELL

options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "Tell me a short story about a dog and a cat. 2 sentences"
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


In [ ]:
# RUN THIS CELL

options = {
    "temperature": 0.00,
    "top_p": 0.001,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "Tell me a short story about a dog and a cat. 2 sentences"
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for _ in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


Compare the two groups of outputs. Look for both **content variation**—changes in characters, events, and wording—and **structural consistency**—whether every response follows the two-sentence constraint.

Because temperature and `top_p` both change between the two blocks, this comparison shows the effect of the combined settings. It does not isolate either parameter by itself.

### TODO - REFLECT 🖊

🖊 **TODO:** Give one business situation in which output variability would be useful and one in which it would create risk or unnecessary review work. Explain why.

🖊 **TODO:** Compare the two code blocks. Which inference settings changed, and in what direction?

🖊 **TODO:** Create a simple quantitative similarity scale—for example, `1 = entirely different` through `5 = nearly identical`. Rate each group of three outputs and justify each rating with specific evidence.

## Example 2: A Subjective One-Word Answer

This prompt requests a one-word answer, making variation immediately visible. However, “the best day of the week” is subjective: the model is not retrieving an objectively correct fact.

Run the broad-sampling configuration first and the narrow-sampling configuration second. Then distinguish **repeatability** from **truth**: a repeated answer is more consistent, but it is not automatically more valid.

**Broader sampling configuration**

In [ ]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.995,
    "top_k": 4000,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "What is the best day of the week? Just give me the day, nothing else. You must pick a day." #instruction + " " + CTO_JOB_POSTING
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


**Narrower sampling configuration**

In [ ]:
# RUN THIS CELL
options = {
    "temperature": 0.0,
    "top_p": 0.005,
    "top_k": 1,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = "What is the best day of the week? Just give me the day, nothing else. You must pick a day." #instruction + " " + CTO_JOB_POSTING
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for _ in range(3):
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))


A model can repeat the same answer because the sampling configuration restricts alternatives. That repeatability may be valuable for standardized business outputs, but it does not turn a subjective judgment into a fact or verify a factual answer against a source.


### TODO - REFLECT 🖊

🖊 **TODO:** Do you agree with any of the answers from the broader-sampling runs? Explain the personal criterion you used.

🖊 **TODO:** Do you agree with the narrower-sampling answer? Did repetition make it more persuasive? Why or why not?

🖊 **TODO:** Explain the difference between consistency and truth. What additional evidence would be needed before treating a model output as a factual business input?

# Part 2: Prompt Structure and Fictional Résumé Analysis

The next examples use a fictional CTO job posting and fictional résumés to examine how prompt structure and sampling settings shape an apparently analytical output.

A numerical score can make an output look objective even when the scoring categories, weights, and interpretations have not been validated. Your task is to evaluate the model's consistency, evidence use, and limitations—not to decide whether a fictional applicant should be hired. In a real employment process, AI could help organize job-relevant evidence for human review, but it should not make or authorize the employment decision.


## Elena Martinez: Hold the Source Constant

The first three comparisons use the same fictional job posting and **Elena Martinez résumé**. Holding those documents constant makes it easier to see what changes when the prompt or sampling configuration changes.


### Example 1: Under-Specified Scoring Prompt

The instruction asks for a score but supplies no scale, categories, weights, or output format. With temperature set to `1.0`, the model also has substantial latitude in how it responds. Run the prompt three times and note any criteria the model invents on its own.

In [ ]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

instruction = "Score the resume against the job posting."
prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_ELENA
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))

#display_record(record)


### TODO - REFLECT 🖊

🖊 **TODO:** How variable are the scores, explanations, and formats from run to run? Cite specific differences.

🖊 **TODO:** Does the model introduce hire/no-hire language even though the prompt asks only for a score? If it does, explain why that is an important scope problem.

🖊 **TODO:** What scoring criteria did the model appear to invent? Were those criteria and their weights consistent across runs and traceable to the job posting?

### Example 2: Add a Structured Scoring Framework

The next prompt defines two scoring categories, assigns weights, and specifies an output format. The model, source documents, and inference settings *remain the same as in Example 1*, so the main change is the **prompt structure**.

Predict which parts of the response will become more consistent. A more structured format may reduce presentation variation, but it does not validate the chosen categories or make the resulting score suitable for an employment decision.

In [ ]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

instruction = """

Score the resume against the job posting.

For your score:
* Use 2 areas, education and experience.
* Education should be weighted at 30 points.
* Experience should be weighted at 70 points.
The total score should be between 0 and 100.

Your output should look as follows:
Total Score:
Education Score:
Experience Score:

Justification for Experience Score: Keep this to 2-3 sentence.
Justification for Education Score: Keep this to 2-3 sentence.

"""



prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_ELENA
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    record = run_once(model, prompt, run_key, options)
    print(f"RUN {i}: ################ ")
    display(Markdown(record['response']['response']))


The structured prompt reduces ambiguity by telling the model how to allocate points and present its response. Compare the outputs with Example 1, focusing separately on format consistency, score consistency, and evidence quality.

### TODO - REFLECT 🖊

🖊 **TODO:** Which parts of the results are more consistent than in Example 1: format, score, evidence, or all three? Cite examples.

🖊 **TODO:** Which additions to the prompt most likely reduced ambiguity? Explain what the prompt structure improved and what it could not validate.


### Example 3: Lower Temperature with the Structured Prompt

Run the same structured prompt again with temperature reduced from `1.0` to `0.0`; all other listed settings remain unchanged. This is a cleaner one-factor comparison than the story demonstration. Compare it directly with the structured high-temperature runs above.


In [ ]:
# RUN THIS CELL
options = {
    "temperature": 0.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_ELENA
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    print(f"RUN {i}: ################ ")
    record = run_once(model, prompt, run_key, options)
    display(Markdown(record['response']['response']))

#display_record(record)


### Example 4: Apply the Structured Prompt to Marcus Reed

The final run returns to the structured high-variation configuration from Example 2 and changes the fictional résumé from Elena Martinez to Marcus Reed. Compare Examples 2 and 4 when asking whether the same prompt and settings respond appropriately to different source evidence. Do not compare Examples 3 and 4 as a one-factor test because both temperature and the source change.

Do not compare the applicants as a hiring exercise. Focus on the model's process: what evidence it selects, whether its arithmetic is coherent, and whether its explanation stays grounded in the supplied documents.

In [ ]:
# RUN THIS CELL
options = {
    "temperature": 1.0,
    "top_p": 0.90,
    "top_k": 40,
    "num_ctx": 4096,
    "num_predict": -1,
}

instruction = """

Score the resume against the job posting.

For your score:
* Use 2 areas, education and experience.
* Education should be weighted at 30 points.
* Experience should be weighted at 70 points.
The total score should be between 0 and 100.

Your output should look as follows:
Total Score:
Education Score:
Experience Score:

Justification for Experience Score: Keep this to 2-3 sentence.
Justification for Education Score: Keep this to 2-3 sentence.

"""



prompt = instruction + " " + CTO_JOB_POSTING + " " + RESUME_MARCUS
model = PRIMARY_MODEL
run_key = "cto_job_posting_summary"

for i in range(3):
    record = run_once(model, prompt, run_key, options)
    print(f"RUN {i}: ################ ")
    display(Markdown(record['response']['response']))


### TODO - REFLECT ON THE MARCUS RESULTS 🖊

Marcus's résumé is intentionally more ambiguous than Elena's: it includes relevant technical and healthcare-software experience, but it does not clearly satisfy several senior leadership requirements. This makes the example especially useful for examining whether the model applies the scoring framework consistently when the evidence is mixed.

🖊 **TODO:** Record the total, education, and experience scores from all three runs. For each score type, calculate the minimum, maximum, and range (`maximum - minimum`). Which category varied most?

🖊 **TODO:** Identify one piece of résumé evidence that the model interpreted or weighted differently across runs. Cite the relevant language from at least two outputs.

🖊 **TODO:** Compare the model's justifications with the actual job posting. Identify one relevant qualification gap the model evaluated consistently and one criterion it introduced, overstated, or weighted inconsistently. For example, check whether an advanced degree is actually required.

🖊 **TODO:** Why might a middle-of-the-road résumé produce more score variation than an obviously strong match? Present your explanation as a hypothesis supported by these outputs—not as a proven rule about all models or applicants.

🖊 **TODO:** Imagine that an organization used a fixed score cutoff. Explain how the observed run-to-run variation could change the outcome for the same person and why this makes model-generated scores unsuitable as hiring decisions.


# Part 3: Final Writeup

### TODO - FINAL WRITEUP 🖊

Write approximately **300–450 words** that synthesizes what you observed across the notebook. Address all of the following:

1. Compare the output variability you observed in the story, day-of-week, and résumé examples. Include the Marcus score ranges and refer to at least two specific outputs from your runs.
2. Explain the difference between what prompt structure appeared to influence and what the inference settings appeared to influence.
3. Recommend a prompt structure and inference settings for one bounded business information-gathering task. Explain the tradeoffs behind your choices.
4. Explain why an apparently consistent résumé score does **not** prove that the model is qualified to make a hiring decision. Describe how the model could instead support evidence gathering while a human remains responsible for the decision.
5. Identify one limitation of this experiment and propose one controlled follow-up test.

**🖊 TODO: Write your final analysis here.**
